<a href="https://colab.research.google.com/github/gayathridevies-arch/Aquatic-Ecotoxicity-Risk-Screener/blob/main/Aquatic_Ecotoxicity_Risk_Screener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, PandasTools
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score

# 1. Define dataset of representative organic pollutants with experimental acute aquatic toxicity (pLC50 / toxicity index)
raw_ecox_data = [
    {"Compound_Name": "Benzene", "SMILES": "c1ccccc1", "pLC50": 2.15},
    {"Compound_Name": "Toluene", "SMILES": "Cc1ccccc1", "pLC50": 2.30},
    {"Compound_Name": "Trichloroethylene (TCE)", "SMILES": "ClC=C(Cl)Cl", "pLC50": 2.50},
    {"Compound_Name": "Carbon Tetrachloride", "SMILES": "ClC(Cl)(Cl)Cl", "pLC50": 2.80},
    {"Compound_Name": "1,4-Dichlorobenzene", "SMILES": "Clc1ccc(Cl)cc1", "pLC50": 3.10},
    {"Compound_Name": "Atrazine", "SMILES": "Clc1nc(nc(n1)NC(C)C)NCC", "pLC50": 3.40},
    {"Compound_Name": "2,4-D", "SMILES": "Clc1cc(Cl)c(cc1)OCC(=O)O", "pLC50": 2.90},
    {"Compound_Name": "4-Nitrophenol", "SMILES": "Oc1ccc(cc1)[N+](=O)[O-]", "pLC50": 3.25},
    {"Compound_Name": "Nitrobenzene", "SMILES": "O=[N+]([O-])c1ccccc1", "pLC50": 2.60},
    {"Compound_Name": "Naphthalene", "SMILES": "c1ccc2ccccc2c1", "pLC50": 3.05},
    {"Compound_Name": "Phenanthrene", "SMILES": "c1ccc2c(c1)ccc3ccccc32", "pLC50": 4.20},
    {"Compound_Name": "Bisphenol A (BPA)", "SMILES": "CC(C)(c1ccc(cc1)O)c2ccc(cc2)O", "pLC50": 3.85},
    {"Compound_Name": "Triclosan", "SMILES": "Clc1cc(c(cc1Cl)O)Oc2ccc(Cl)cc2Cl", "pLC50": 4.90},
    {"Compound_Name": "Phenol", "SMILES": "Oc1ccccc1", "pLC50": 2.40},
    {"Compound_Name": "Aniline", "SMILES": "Nc1ccccc1", "pLC50": 2.20},
    {"Compound_Name": "Formaldehyde", "SMILES": "C=O", "pLC50": 1.50},
    {"Compound_Name": "Ethanol", "SMILES": "CCO", "pLC50": 0.80},
    {"Compound_Name": "Glucose", "SMILES": "OC[C@H]1O[C@H](O)[C@H](O)[C@@H](O)[C@H]1O", "pLC50": 0.30},
    {"Compound_Name": "Naphthalene, 2-chloro-", "SMILES": "Clc1ccc2ccccc2c1", "pLC50": 3.55},
    {"Compound_Name": "Pentachlorophenol", "SMILES": "Oc1c(Cl)c(Cl)c(Cl)c(Cl)c1Cl", "pLC50": 5.10}
]

df = pd.DataFrame(raw_ecox_data)
PandasTools.AddMoleculeColumnToFrame(df, smilesCol='SMILES', molCol='Molecule')

def extract_descriptors(mol):
    if mol is None:
        return None
    return {
        "Molecular_Weight": Descriptors.MolWt(mol),"LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),"Num_H_Donors": Descriptors.NumHDonors(mol),"Num_H_Acceptors": Descriptors.NumHAcceptors(mol)}
descriptor_list = df['Molecule'].apply(extract_descriptors)
desc_df = pd.DataFrame(list(descriptor_list))
df = pd.concat([df, desc_df], axis=1)

# Drop rows with parsing errors if any
df = df.dropna(subset=['LogP'])

# 3. Prepare data for Machine Learning (QSAR Regression)
feature_cols = ["Molecular_Weight", "LogP", "TPSA", "Num_H_Donors", "Num_H_Acceptors"]
X = df[feature_cols]
y = df["pLC50"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
loo = LeaveOneOut()
y_pred_cv = np.zeros_like(y, dtype=float)

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred_cv[test_idx] = model.predict(X_test)[0]
cv_r2 = r2_score(y, y_pred_cv)
cv_rmse = np.sqrt(mean_squared_error(y, y_pred_cv))

print(f"LOOCV R² Score: {cv_r2:.3f}")
print(f"LOOCV RMSE: {cv_rmse:.3f}")
model.fit(X, y)
df["Predicted_pLC50"] = model.predict(X)
df.drop(columns=['Molecule']).to_csv("aquatic_risk_predictions.csv", index=False)
print("Saved predictions to aquatic_risk_predictions.csv")

LOOCV R² Score: 0.589
LOOCV RMSE: 0.748
Saved predictions to aquatic_risk_predictions.csv
